# Normalizing Flow (1D Affine)

Chapter 13: Probability and Random Variables · *Mathematics for AI and Machine Learning* by Fuheng Wu

Chain two affine layers on N(0,1); exact density via cumulative log|det J|.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-13/normalizing-flow/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
from torch.distributions import Normal

torch.manual_seed(0)

# The demo's defaults: two affine layers z -> exp(s) z + t on top of N(0, 1)
s1, t1, s2, t2 = 0.4, 0.0, -0.3, 0.5
base = Normal(0.0, 1.0)


def forward(z0):
    z1 = math.exp(s1) * z0 + t1
    return math.exp(s2) * z1 + t2


def inverse(z2):
    z1 = (z2 - t2) * math.exp(-s2)
    return (z1 - t1) * math.exp(-s1)


def log_q(z2):
    """log q(z2) = log q0(z0) - sum of log|det J| = log q0(z0) - s1 - s2."""
    return base.log_prob(inverse(z2)) - (s1 + s2)


# ── Exact density through the flow ──────────────────────────────────────────
z0 = torch.randn(400000, dtype=torch.float64)
z2 = forward(z0)
print("total log|det J| of the forward map =", s1 + s2)
print("invert(forward(z0)) == z0:", torch.allclose(inverse(z2), z0))

grid = torch.linspace(-6, 6, 4001, dtype=torch.float64)
dens = log_q(grid).exp()
print("q integrates to", torch.trapezoid(dens, grid).item())

# Two affine layers are one affine map: z2 = exp(s1 + s2) z0 + (exp(s2) t1 + t2)
scale, shift = math.exp(s1 + s2), math.exp(s2) * t1 + t2
print(f"so q = N({shift:.4f}, {scale:.4f}^2);  sample mean {z2.mean().item():.4f}, sample std {z2.std().item():.4f}")
print("log q(1.0) via the flow =", log_q(torch.tensor(1.0, dtype=torch.float64)).item(),
      " via Normal(shift, scale) =", Normal(shift, scale).log_prob(torch.tensor(1.0)).item())

# ── Training a one-layer flow by maximum likelihood recovers the data law ───
data = 2.0 + 0.5 * torch.randn(5000, dtype=torch.float64)       # N(2, 0.5^2)
s = torch.zeros((), dtype=torch.float64, requires_grad=True)
t = torch.zeros((), dtype=torch.float64, requires_grad=True)
opt = torch.optim.Adam([s, t], lr=0.05)
for _ in range(600):
    opt.zero_grad()
    nll = -(base.log_prob((data - t) * torch.exp(-s)) - s).mean()
    nll.backward()
    opt.step()
print(f"\nfitted flow: shift t = {t.item():.3f} (2.0), scale exp(s) = {s.exp().item():.3f} (0.5)")

total log|det J| of the forward map = 0.10000000000000003
invert(forward(z0)) == z0: True
q integrates to 0.9999996744167559
so q = N(0.5000, 1.1052^2);  sample mean 0.4987, sample std 1.1033
log q(1.0) via the flow = -1.1212798773394206  via Normal(shift, scale) = -1.1212799549102783



fitted flow: shift t = 2.014 (2.0), scale exp(s) = 0.496 (0.5)
